<!-- Cell 1 -->
# AquaSelect -- Notebook 2a: Selective Prediction Framework (Frozen Backbone)

**Purpose**: Evaluate 4 selection methods (SR, MC Dropout, Deep Ensemble, AquaSelect) on both ConvNeXt-Tiny and DeiT-Small backbones with a **frozen backbone**.

**Inputs**: Trained checkpoints + saved logits from Notebooks 1a/1b, AQUA20 dataset.

**Output**: Risk-coverage curves, AURCC, Coverage@95, Coverage@99 for all methods.

**Hardware**: Kaggle T4 GPU (16GB VRAM)

**Backbone**: Frozen -- only the selection head is trained.

In [ ]:
# Cell 2
!pip install -q timm datasets pyarrow

In [ ]:
# Cell 3
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

import timm
import cv2
from torchvision import transforms
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 4
# Configuration 
CFG = {
    "num_classes": 20,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 2,
    "seeds": [9, 42, 2003],
    "split_seed": 42,
    "val_size": 0.15,
    # MCD settings
    "mcd_forward_passes": 10,
    "mcd_dropout_rate": 0.3,
    # AquaSelect selection head training
    "sel_epochs": 15,
    "sel_lr": 1e-3,
    "sel_patience": 5,
    "sel_alpha": 0.5,       # abstention cost
    "sel_lambda": 1.0,      # selection loss weight
    # Paths
    "data_path": "/kaggle/input/aqua20",
    "convnext_path": "/kaggle/input/datasets/abcd/aqua20-convnext-models",
    "deit_path": "/kaggle/input/datasets/abcd/aqua20-deit-models",
    "output_dir": "/kaggle/working",
}

# Backbone configs
BACKBONES = {
    "convnext_tiny": {
        "timm_name": "convnext_tiny",
        "feature_dim": 768,
        "ckpt_path": CFG["convnext_path"],
        "logits_file": "convnext_val_test_outputs.pth",
        "ckpt_pattern": "convnext_tiny_seed_{seed}.pth",
    },
    "deit_small": {
        "timm_name": "deit_small_patch16_224",
        "feature_dim": 384,
        "ckpt_path": CFG["deit_path"],
        "logits_file": "deit_val_test_outputs.pth",
        "ckpt_pattern": "deit_small_seed_{seed}.pth",
    },
}

print("Configuration:")
for k, v in CFG.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 5
# Load AQUA20 and recreate identical splits 
dataset = load_dataset(
    "parquet",
    data_files={
        "train": "/kaggle/input/datasets/abcd/aqua-20/train-00000-of-00001.parquet",
        "test":  "/kaggle/input/datasets/abcd/aqua-20/test-00000-of-00001.parquet",
    }
)

train_full = dataset["train"]
test_ds = dataset["test"]
all_labels = train_full["label"]
label_names = train_full.features["label"].names

# Recreate identical stratified split
train_indices, val_indices = train_test_split(
    range(len(train_full)),
    test_size=CFG["val_size"],
    random_state=CFG["split_seed"],
    stratify=all_labels,
)

print(f"Train subset: {len(train_indices)} | Val subset: {len(val_indices)} | Test: {len(test_ds)}")
print(f"Classes: {len(label_names)}")

In [ ]:
# Cell 6
# Dataset class and dataloaders 

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CFG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# need raw images (no normalization) for UDAS features
raw_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CFG["image_size"]),
])


class AQUA20Dataset(Dataset):
    def __init__(self, hf_dataset, indices=None, transform=None):
        self.dataset = hf_dataset
        self.indices = indices if indices is not None else list(range(len(hf_dataset)))
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        item = self.dataset[self.indices[idx]]
        image = item["image"].convert("RGB")
        label = item["label"]
        if self.transform:
            image = self.transform(image)
        return image, label


class AQUA20RawDataset(Dataset):
    """Returns raw numpy images (for UDAS feature extraction)."""
    def __init__(self, hf_dataset, indices=None, transform=None):
        self.dataset = hf_dataset
        self.indices = indices if indices is not None else list(range(len(hf_dataset)))
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        item = self.dataset[self.indices[idx]]
        image = item["image"].convert("RGB")
        if self.transform:
            image = self.transform(image)
        return np.array(image), item["label"]


# Standard eval dataloaders
val_dataset = AQUA20Dataset(train_full, val_indices, eval_transform)
test_dataset = AQUA20Dataset(test_ds, None, eval_transform)
train_sub_dataset = AQUA20Dataset(train_full, train_indices, eval_transform)

val_loader = DataLoader(val_dataset, batch_size=CFG["batch_size"], shuffle=False,
                        num_workers=CFG["num_workers"], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CFG["batch_size"], shuffle=False,
                         num_workers=CFG["num_workers"], pin_memory=True)

# For selection head training (with augmentation)
train_aug_transform = transforms.Compose([
    transforms.RandomResizedCrop(CFG["image_size"], scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_aug_dataset = AQUA20Dataset(train_full, train_indices, train_aug_transform)
train_aug_loader = DataLoader(train_aug_dataset, batch_size=CFG["batch_size"], shuffle=True,
                              num_workers=CFG["num_workers"], pin_memory=True)

print(f"Val loader: {len(val_loader)} batches")
print(f"Test loader: {len(test_loader)} batches")
print(f"Train aug loader: {len(train_aug_loader)} batches")

In [ ]:
# Cell 7
# Load saved logits 

saved_logits = {}

for backbone_name, bcfg in BACKBONES.items():
    logits_file = os.path.join(bcfg["ckpt_path"], bcfg["logits_file"])
    data = torch.load(logits_file, map_location="cpu", weights_only=False)
    saved_logits[backbone_name] = data
    print(f"\n{backbone_name}:")
    print(f"  Val logits:  {data['val_logits'][42].shape}")
    print(f"  Test logits: {data['test_logits'][42].shape}")
    print(f"  Val labels:  {data['val_labels'].shape}")
    print(f"  Test labels: {data['test_labels'].shape}")
    for seed in CFG["seeds"]:
        preds = data["test_logits"][seed].argmax(dim=1)
        acc = (preds == data["test_labels"]).float().mean().item() * 100
        print(f"  Seed {seed} test acc: {acc:.2f}%")

print("\nAll logits loaded successfully.")

In [ ]:
# Cell 8
# Model factory for loading checkpoints 

class BackboneClassifier(nn.Module):
    def __init__(self, backbone, classifier):
        super().__init__()
        self.backbone = backbone
        self.classifier = classifier

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits, features


def load_trained_model(backbone_name, seed):
    """Load a trained backbone+classifier from checkpoint."""
    bcfg = BACKBONES[backbone_name]
    ckpt_file = os.path.join(bcfg["ckpt_path"],
                             bcfg["ckpt_pattern"].format(seed=seed))
    ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)

    backbone = timm.create_model(bcfg["timm_name"], pretrained=False, num_classes=0)
    classifier = nn.Linear(bcfg["feature_dim"], CFG["num_classes"])
    model = BackboneClassifier(backbone, classifier)
    model.load_state_dict(ckpt["model_state_dict"])
    return model


#  test: load one model
_model = load_trained_model("convnext_tiny", 42)
print(f"Loaded convnext_tiny seed 42: {sum(p.numel() for p in _model.parameters())/1e6:.1f}M params")
del _model

In [ ]:
# Cell 9
# Selective prediction metrics 

def compute_risk_coverage_curve(predictions, targets, confidence_scores):
    """Compute risk-coverage curve.
    
    Args:
        predictions: (N,) predicted class labels
        targets: (N,) true class labels  
        confidence_scores: (N,) higher = more confident
    
    Returns:
        coverages: list of coverage values
        risks: list of risk (error rate) values
        aurcc: area under risk-coverage curve (lower is better)
    """
    sorted_indices = np.argsort(-confidence_scores)  # descending confidence
    sorted_correct = (predictions == targets)[sorted_indices]

    n = len(sorted_correct)
    coverages = []
    risks = []

    for k in range(1, n + 1):
        cov = k / n
        risk = 1.0 - sorted_correct[:k].mean()
        coverages.append(cov)
        risks.append(risk)

    aurcc = np.trapz(risks, coverages)
    return np.array(coverages), np.array(risks), aurcc


def coverage_at_accuracy(predictions, targets, confidence_scores, target_acc=0.95):
    """Find maximum coverage where selective accuracy >= target_acc."""
    sorted_indices = np.argsort(-confidence_scores)
    sorted_correct = (predictions == targets)[sorted_indices]
    n = len(sorted_correct)
    best_coverage = 0.0
    for k in range(1, n + 1):
        acc = sorted_correct[:k].mean()
        if acc >= target_acc:
            best_coverage = k / n
    return best_coverage * 100  # return as percentage


def evaluate_selection_method(predictions, targets, confidence_scores, method_name=""):
    """Compute all selective prediction metrics for a method."""
    coverages, risks, aurcc = compute_risk_coverage_curve(predictions, targets, confidence_scores)
    cov95 = coverage_at_accuracy(predictions, targets, confidence_scores, target_acc=0.95)
    cov99 = coverage_at_accuracy(predictions, targets, confidence_scores, target_acc=0.99)
    
    # Full coverage accuracy (standard)
    full_acc = (predictions == targets).mean() * 100
    
    return {
        "method": method_name,
        "full_acc": full_acc,
        "aurcc": aurcc,
        "cov95": cov95,
        "cov99": cov99,
        "coverages": coverages,
        "risks": risks,
    }


print("Selective prediction metrics defined.")

In [ ]:
# Cell 10
# Method 1: Softmax Response (SR) 
# Uses max(softmax) as confidence.

def run_softmax_response(logits, labels):
    """Softmax Response: max(softmax(logits)) as confidence score."""
    probs = F.softmax(logits, dim=1)
    confidence = probs.max(dim=1).values.numpy()
    predictions = logits.argmax(dim=1).numpy()
    targets = labels.numpy()
    return predictions, targets, confidence


print("--- Softmax Response (SR) ---")
sr_results = {}

for backbone_name in BACKBONES:
    sr_results[backbone_name] = {}
    data = saved_logits[backbone_name]
    
    for seed in CFG["seeds"]:
        test_logits = data["test_logits"][seed]
        test_labels = data["test_labels"]
        preds, targets, conf = run_softmax_response(test_logits, test_labels)
        result = evaluate_selection_method(preds, targets, conf, method_name="SR")
        sr_results[backbone_name][seed] = result
        print(f"  {backbone_name} seed {seed}: AURCC={result['aurcc']:.4f} "
              f"Cov@95={result['cov95']:.1f}% Cov@99={result['cov99']:.1f}%")

print("\nSR complete for all backbones and seeds.")

In [ ]:
# Cell 11
# Method 2: Deep Ensemble (DE) 
# Average softmax across 3 seed models, use entropy of averaged distribution.

def run_deep_ensemble(logits_dict, labels, seeds):
    """Deep Ensemble: average softmax across seeds, use negative entropy as confidence."""
    # Average softmax probabilities across seeds
    avg_probs = torch.zeros_like(F.softmax(logits_dict[seeds[0]], dim=1))
    for seed in seeds:
        avg_probs += F.softmax(logits_dict[seed], dim=1)
    avg_probs /= len(seeds)
    
    # Predictions from averaged probabilities
    predictions = avg_probs.argmax(dim=1).numpy()
    
    # Negative entropy as confidence (higher = more confident)
    entropy = -(avg_probs * torch.log(avg_probs + 1e-8)).sum(dim=1)
    confidence = -entropy.numpy()  # negate so higher = more confident
    
    targets = labels.numpy()
    return predictions, targets, confidence


print("--- Deep Ensemble (DE) ---")
de_results = {}

for backbone_name in BACKBONES:
    data = saved_logits[backbone_name]
    test_labels = data["test_labels"]
    preds, targets, conf = run_deep_ensemble(data["test_logits"], test_labels, CFG["seeds"])
    result = evaluate_selection_method(preds, targets, conf, method_name="DE")
    de_results[backbone_name] = result
    print(f"  {backbone_name}: AURCC={result['aurcc']:.4f} "
          f"Cov@95={result['cov95']:.1f}% Cov@99={result['cov99']:.1f}% "
          f"Full Acc={result['full_acc']:.2f}%")

print("\nDE complete. Note: DE has one result per backbone (ensembles all 3 seeds).")

In [ ]:
# Cell 12
# Method 3: Monte Carlo Dropout (MCD) 
# Load checkpoint, inject dropout before classifier, run 10 forward passes with
# dropout enabled, use negative predictive entropy as confidence.

class BackboneClassifierMCD(nn.Module):
    """Backbone + dropout + classifier for MC Dropout inference."""
    def __init__(self, backbone, classifier, dropout_rate=0.3):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = classifier

    def forward(self, x):
        features = self.backbone(x)
        features_dropped = self.dropout(features)
        logits = self.classifier(features_dropped)
        return logits, features


def run_mcd_inference(model, loader, n_passes=10):
    """Run MC Dropout: n forward passes with dropout enabled.
    
    Returns:
        avg_probs: (N, C) averaged softmax
        predictions: (N,) from avg_probs
        confidence: (N,) negative entropy of avg_probs
        all_labels: (N,)
    """
    model.eval()
    # Enable dropout during inference
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            batch_probs = torch.zeros(images.size(0), CFG["num_classes"]).to(device)
            
            for _ in range(n_passes):
                logits, _ = model(images)
                batch_probs += F.softmax(logits, dim=1)
            
            batch_probs /= n_passes
            all_probs.append(batch_probs.cpu())
            all_labels.append(labels)

    avg_probs = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    
    predictions = avg_probs.argmax(dim=1).numpy()
    entropy = -(avg_probs * torch.log(avg_probs + 1e-8)).sum(dim=1)
    confidence = -entropy.numpy()
    targets = all_labels.numpy()
    
    return predictions, targets, confidence


print("--- MC Dropout (MCD) ---")
mcd_results = {}

for backbone_name, bcfg in BACKBONES.items():
    mcd_results[backbone_name] = {}
    
    for seed in CFG["seeds"]:
        print(f"  {backbone_name} seed {seed}: loading model...", end=" ")
        
        # Load trained model
        base_model = load_trained_model(backbone_name, seed)
        
        # Wrap with MCD dropout
        mcd_model = BackboneClassifierMCD(
            base_model.backbone, base_model.classifier,
            dropout_rate=CFG["mcd_dropout_rate"]
        ).to(device)
        
        # Run MCD inference on test set
        preds, targets, conf = run_mcd_inference(
            mcd_model, test_loader, n_passes=CFG["mcd_forward_passes"]
        )
        
        result = evaluate_selection_method(preds, targets, conf, method_name="MCD")
        mcd_results[backbone_name][seed] = result
        print(f"AURCC={result['aurcc']:.4f} Cov@95={result['cov95']:.1f}% "
              f"Cov@99={result['cov99']:.1f}%")
        
        del mcd_model, base_model
        torch.cuda.empty_cache()

print("\nMCD complete for all backbones and seeds.")

In [ ]:
# Cell 13
# AquaSelect: Selection Head + Selective Loss 

class SelectionHead(nn.Module):
    def __init__(self, feature_dim, hidden_dim=256):
        super().__init__()
        self.selector = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, features):
        return self.selector(features).squeeze(-1)


class AquaSelectModel(nn.Module):
    """Full AquaSelect: frozen backbone + frozen classifier + trainable selection head."""
    def __init__(self, backbone, classifier, feature_dim):
        super().__init__()
        self.backbone = backbone
        self.classifier = classifier
        self.selection_head = SelectionHead(feature_dim)
        
        # Freeze backbone and classifier
        for param in self.backbone.parameters():
            param.requires_grad = False
        for param in self.classifier.parameters():
            param.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
            logits = self.classifier(features)
        sel_score = self.selection_head(features.detach())
        return logits, sel_score, features.detach()


class SelectiveLoss(nn.Module):
    def __init__(self, alpha=0.5, lam=1.0):
        super().__init__()
        self.alpha = alpha   # abstention cost
        self.lam = lam       # selection loss weight
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, sel_scores, targets):
        ce_loss = self.ce(logits, targets)
        select_loss = (
            sel_scores * ce_loss +
            self.alpha * (1.0 - sel_scores)
        ).mean()
        total_loss = ce_loss.mean() + self.lam * select_loss
        return total_loss


print("AquaSelect modules defined.")

In [ ]:
# Cell 14
# Temperature Scaling (post-hoc calibration) 

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    def fit(self, val_logits, val_labels, max_iter=50, lr=0.01):
        """Fit temperature on validation set by minimizing NLL."""
        val_logits = val_logits.to(device)
        val_labels = val_labels.to(device)
        self.to(device)
        nll = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS([self.temperature], lr=lr, max_iter=max_iter)

        def closure():
            optimizer.zero_grad()
            loss = nll(self.forward(val_logits), val_labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        
        # Verify calibration improved
        with torch.no_grad():
            before_nll = nll(val_logits, val_labels).item()
            after_nll = nll(self.forward(val_logits), val_labels).item()
        
        print(f"    Temperature: {self.temperature.item():.4f} | "
              f"NLL before: {before_nll:.4f} -> after: {after_nll:.4f}")
        return self.temperature.item()


print("Temperature Scaler defined.")

In [ ]:
# Cell 15
# UDAS: Underwater Difficulty-Aware Selection features 

def compute_visibility_score(image_np):
    """Estimate underwater visibility via Fourier high-frequency ratio.
    Higher = clearer image = easier to classify."""
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    f_transform = np.fft.fft2(gray)
    f_shift = np.fft.fftshift(f_transform)
    magnitude = np.abs(f_shift)
    h, w = gray.shape
    cy, cx = h // 2, w // 2
    radius = min(h, w) // 8
    Y, X = np.ogrid[:h, :w]
    mask = (Y - cy)**2 + (X - cx)**2 <= radius**2
    low_energy = magnitude[mask].sum()
    total_energy = magnitude.sum() + 1e-8
    high_freq_ratio = 1.0 - (low_energy / total_energy)
    return high_freq_ratio


def compute_color_cast_score(image_np):
    """Measure color cast severity. Higher = more cast = harder.
    Normalized by dataset-empirical max."""
    means = image_np.mean(axis=(0, 1))
    overall_mean = means.mean()
    deviation = np.sqrt(((means - overall_mean) ** 2).sum())
    return deviation


def extract_udas_features(hf_dataset, indices, transform):
    """Extract UDAS features for a set of images.
    
    Returns:
        visibility: (N,) float array
        color_cast: (N,) float array (raw, normalize later)
    """
    visibility = []
    color_cast = []
    
    for i, idx in enumerate(indices):
        item = hf_dataset[idx]
        img = item["image"].convert("RGB")
        if transform:
            img = transform(img)
        img_np = np.array(img)
        
        visibility.append(compute_visibility_score(img_np))
        color_cast.append(compute_color_cast_score(img_np))
        
        if (i + 1) % 500 == 0:
            print(f"    Processed {i+1}/{len(indices)} images...", end="\r")
    
    visibility = np.array(visibility)
    color_cast = np.array(color_cast)
    print(f"    Processed {len(indices)}/{len(indices)} images.    ")
    return visibility, color_cast


# Extract UDAS features for val and test sets
print("Extracting UDAS features for val set...")
val_visibility, val_color_cast = extract_udas_features(train_full, val_indices, raw_transform)

print("Extracting UDAS features for test set...")
test_visibility, test_color_cast = extract_udas_features(test_ds, list(range(len(test_ds))), raw_transform)

# Normalize color cast using val set statistics (99th percentile)
cc_99 = np.percentile(np.concatenate([val_color_cast, test_color_cast]), 99)
val_color_cast_norm = np.clip(val_color_cast / (cc_99 + 1e-8), 0, 1)
test_color_cast_norm = np.clip(test_color_cast / (cc_99 + 1e-8), 0, 1)

print(f"\nUDAS feature stats:")
print(f"  Val  visibility: {val_visibility.mean():.4f} +/- {val_visibility.std():.4f}")
print(f"  Val  color_cast: {val_color_cast_norm.mean():.4f} +/- {val_color_cast_norm.std():.4f}")
print(f"  Test visibility: {test_visibility.mean():.4f} +/- {test_visibility.std():.4f}")
print(f"  Test color_cast: {test_color_cast_norm.mean():.4f} +/- {test_color_cast_norm.std():.4f}")
print(f"  Color cast 99th percentile (normalization): {cc_99:.2f}")

In [ ]:
# Cell 16
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_selection_head(backbone_name, seed, train_loader, val_loader, cfg):
    """Train selection head as binary classifier: correct(1) vs incorrect(0).
    
    Backbone and classifier are fully frozen. Only the selection head trains.
    Binary target: does the frozen classifier predict the correct class?
    
    Returns:
        model: AquaSelectModel with trained selection head
        val_sel_scores: (N_val,) selection scores on val set
        test_sel_scores: (N_test,) selection scores on test set
    """
    bcfg = BACKBONES[backbone_name]
    set_seed(seed)
    
    # Load trained backbone+classifier
    base_model = load_trained_model(backbone_name, seed)
    
    # Build AquaSelect model (freezes backbone+classifier)
    model = AquaSelectModel(
        base_model.backbone, base_model.classifier, bcfg["feature_dim"]
    ).to(device)
    
    # Only selection head parameters are trainable
    trainable_params = list(model.selection_head.parameters())
    print(f"    Trainable params: {sum(p.numel() for p in trainable_params)/1e3:.1f}K")
    
    optimizer = AdamW(trainable_params, lr=cfg["sel_lr"], weight_decay=1e-4)
    bce_loss_fn = nn.BCELoss()
    
    best_val_loss = float("inf")
    best_val_auroc = 0.0
    patience_counter = 0
    best_state = None
    
    for epoch in range(1, cfg["sel_epochs"] + 1):
        # Train 
        model.selection_head.train()
        running_loss = 0
        train_correct_sel = 0
        train_total = 0
        num_batches = len(train_loader)
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            logits, sel_scores, _ = model(images)
            
            # Binary target
            with torch.no_grad():
                preds = logits.argmax(dim=1)
                binary_target = (preds == labels).float()
            
            loss = bce_loss_fn(sel_scores, binary_target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * labels.size(0)
            # Track selection head accuracy
            sel_preds = (sel_scores > 0.5).float()
            train_correct_sel += (sel_preds == binary_target).sum().item()
            train_total += labels.size(0)
        
        train_loss = running_loss / train_total
        train_sel_acc = train_correct_sel / train_total * 100
        
        # Validate 
        model.eval()
        val_loss = 0
        val_correct_sel = 0
        val_total = 0
        val_scores_list = []
        val_targets_list = []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                logits, sel_scores, _ = model(images)
                
                preds = logits.argmax(dim=1)
                binary_target = (preds == labels).float()
                
                loss = bce_loss_fn(sel_scores, binary_target)
                val_loss += loss.item() * labels.size(0)
                
                sel_preds = (sel_scores > 0.5).float()
                val_correct_sel += (sel_preds == binary_target).sum().item()
                val_total += labels.size(0)
                
                val_scores_list.append(sel_scores.cpu())
                val_targets_list.append(binary_target.cpu())
        
        val_loss /= val_total
        val_sel_acc = val_correct_sel / val_total * 100
        
        improved = ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = copy.deepcopy(model.selection_head.state_dict())
            improved = " *"
        else:
            patience_counter += 1
        
        print(f"    Epoch {epoch}/{cfg['sel_epochs']} | "
              f"Train Loss: {train_loss:.4f} Sel Acc: {train_sel_acc:.1f}% | "
              f"Val Loss: {val_loss:.4f} Sel Acc: {val_sel_acc:.1f}% | "
              f"Pat: {patience_counter}/{cfg['sel_patience']}{improved}")
        
        if patience_counter >= cfg["sel_patience"]:
            print(f"    Early stopping at epoch {epoch}")
            break
    
    print(f"    Best val loss: {best_val_loss:.4f}")
    
    # Restore best selection head
    model.selection_head.load_state_dict(best_state)
    model.eval()
    
    # Extract selection scores on val and test sets 
    def get_sel_scores(loader):
        all_scores = []
        with torch.no_grad():
            for images, labels in loader:
                images = images.to(device)
                _, sel_scores, _ = model(images)
                all_scores.append(sel_scores.cpu())
        return torch.cat(all_scores).numpy()
    
    val_sel_scores = get_sel_scores(val_loader)
    test_sel_scores = get_sel_scores(test_loader)
    
    # Report how discriminative the selection head is
    val_scores_all = torch.cat(val_scores_list).numpy()
    val_targets_all = torch.cat(val_targets_list).numpy()
    correct_mask = val_targets_all == 1
    print(f"    Val sel score (correct preds):   {val_scores_all[correct_mask].mean():.3f} "
          f"+/- {val_scores_all[correct_mask].std():.3f}")
    print(f"    Val sel score (incorrect preds): {val_scores_all[~correct_mask].mean():.3f} "
          f"+/- {val_scores_all[~correct_mask].std():.3f}")
    print(f"    Separation: {val_scores_all[correct_mask].mean() - val_scores_all[~correct_mask].mean():.3f}")
    
    return model, val_sel_scores, test_sel_scores


print("Selection head training function defined (binary correct/incorrect).")

In [ ]:
# Cell 17
# AquaSelect: UDAS fusion (logistic regression on val, apply to test) 

def compute_aquaselect_scores(
    val_sel_scores, test_sel_scores,
    val_logits, test_logits,
    val_visibility, test_visibility,
    val_color_cast, test_color_cast,
    temperature_scaler,
    val_labels,
):
    """Fuse selection head score + calibrated max prob + UDAS features.
    
    Fit fusion weights via logistic regression on val set (target: correct/incorrect).
    Apply to test set.
    
    Returns:
        test_confidence: (N_test,) final fused confidence scores
        val_confidence: (N_val,) for analysis
    """
    # Calibrated max prob
    with torch.no_grad():
        val_cal_probs = F.softmax(temperature_scaler(val_logits.to(device)), dim=1).cpu()
        test_cal_probs = F.softmax(temperature_scaler(test_logits.to(device)), dim=1).cpu()
    val_max_prob = val_cal_probs.max(dim=1).values.numpy()
    test_max_prob = test_cal_probs.max(dim=1).values.numpy()
    
    # Build feature matrices
    val_features = np.stack([
        val_sel_scores,
        val_max_prob,
        val_visibility,
        1.0 - val_color_cast,  # invert: low cast = easier = higher score
    ], axis=1)
    
    test_features = np.stack([
        test_sel_scores,
        test_max_prob,
        test_visibility,
        1.0 - test_color_cast,
    ], axis=1)
    
    # Target: was the prediction correct?
    val_preds = val_logits.argmax(dim=1).numpy()
    val_correct = (val_preds == val_labels.numpy()).astype(int)
    
    # Fit logistic regression
    lr_model = LogisticRegression(max_iter=1000, random_state=42)
    lr_model.fit(val_features, val_correct)
    
    # Predict probability of being correct
    val_confidence = lr_model.predict_proba(val_features)[:, 1]
    test_confidence = lr_model.predict_proba(test_features)[:, 1]
    
    # Print fusion weights
    print(f"    UDAS fusion weights: {lr_model.coef_[0]}")
    print(f"    Features: [sel_score, cal_max_prob, visibility, 1-color_cast]")
    print(f"    Val accuracy of fusion model: {lr_model.score(val_features, val_correct)*100:.1f}%")
    
    return test_confidence, val_confidence, lr_model


print("UDAS fusion function defined.")

In [ ]:
# Cell 18
# Run AquaSelect: selection head (binary) + temperature scaling + UDAS fusion 

print("="*70)
print("  AQUASELECT -- Full Pipeline (Binary Selection Head)")
print("="*70)

aquaselect_results = {}

for backbone_name, bcfg in BACKBONES.items():
    aquaselect_results[backbone_name] = {}
    data = saved_logits[backbone_name]
    
    for seed in CFG["seeds"]:
        print(f"\n--- {backbone_name} | seed {seed} ---")
        
        # Step 1: Train selection head (binary correct/incorrect)
        print("  [1/3] Training selection head (binary BCE)...")
        model, val_sel_scores, test_sel_scores = train_selection_head(
            backbone_name, seed, train_aug_loader, val_loader, CFG
        )
        
        # Step 2: Fit temperature scaling on val logits
        print("  [2/3] Fitting temperature scaling...")
        temp_scaler = TemperatureScaler()
        val_logits = data["val_logits"][seed]
        val_labels = data["val_labels"]
        temp_scaler.fit(val_logits, val_labels)
        
        # Step 3: UDAS fusion
        print("  [3/3] UDAS fusion...")
        test_logits = data["test_logits"][seed]
        test_labels = data["test_labels"]
        
        test_confidence, val_confidence, fusion_model = compute_aquaselect_scores(
            val_sel_scores, test_sel_scores,
            val_logits, test_logits,
            val_visibility, test_visibility,
            val_color_cast_norm, test_color_cast_norm,
            temp_scaler, val_labels,
        )
        
        # Evaluate
        predictions = test_logits.argmax(dim=1).numpy()
        targets = test_labels.numpy()
        result = evaluate_selection_method(predictions, targets, test_confidence,
                                           method_name="AquaSelect")
        aquaselect_results[backbone_name][seed] = result
        
        # Store extra data for ablation and downstream analysis
        result["val_sel_scores"] = val_sel_scores
        result["test_sel_scores"] = test_sel_scores
        result["test_confidence"] = test_confidence
        result["temperature"] = temp_scaler.temperature.item()
        result["temp_scaler"] = temp_scaler
        result["fusion_model"] = fusion_model
        
        print(f"  >> AURCC={result['aurcc']:.4f} | "
              f"Cov@95={result['cov95']:.1f}% | Cov@99={result['cov99']:.1f}%")
        
        # show how AquaSelect compares to SR for this seed
        sr_aurcc = sr_results[backbone_name][seed]["aurcc"]
        delta = result["aurcc"] - sr_aurcc
        direction = "worse" if delta > 0 else "better"
        print(f"  >> vs SR: AURCC delta = {delta:+.4f} ({direction})")
        
        # Cleanup
        del model
        torch.cuda.empty_cache()

print(f"\n{'='*70}")
print("AquaSelect complete for all backbones and seeds.")

In [ ]:
# Cell 19
# Aggregate results

def aggregate_method_results(results_dict, method_name, seeds):
    """Aggregate per-seed results into mean +/- std.
    For DE, there's only one result (no per-seed), handle that."""
    if isinstance(results_dict, dict) and "aurcc" in results_dict:
        # Single result (DE)
        return {
            "method": method_name,
            "aurcc_mean": results_dict["aurcc"],
            "aurcc_std": 0.0,
            "cov95_mean": results_dict["cov95"],
            "cov95_std": 0.0,
            "cov99_mean": results_dict["cov99"],
            "cov99_std": 0.0,
        }
    else:
        # Per-seed results
        aurccs = [results_dict[s]["aurcc"] for s in seeds]
        cov95s = [results_dict[s]["cov95"] for s in seeds]
        cov99s = [results_dict[s]["cov99"] for s in seeds]
        return {
            "method": method_name,
            "aurcc_mean": np.mean(aurccs),
            "aurcc_std": np.std(aurccs),
            "cov95_mean": np.mean(cov95s),
            "cov95_std": np.std(cov95s),
            "cov99_mean": np.mean(cov99s),
            "cov99_std": np.std(cov99s),
        }


for backbone_name in BACKBONES:
    print(f"\n{'='*70}")
    print(f"  TABLE: Selective Prediction Results -- {backbone_name}")
    print(f"{'='*70}")
    
    rows = []
    rows.append(aggregate_method_results(sr_results[backbone_name], "SR", CFG["seeds"]))
    rows.append(aggregate_method_results(mcd_results[backbone_name], "MCD", CFG["seeds"]))
    rows.append(aggregate_method_results(de_results[backbone_name], "DE", CFG["seeds"]))
    rows.append(aggregate_method_results(aquaselect_results[backbone_name], "AquaSelect", CFG["seeds"]))
    
    df = pd.DataFrame(rows)
    
    # Format for display
    display_rows = []
    for _, row in df.iterrows():
        display_rows.append({
            "Method": row["method"],
            "AURCC": f"{row['aurcc_mean']:.4f} +/- {row['aurcc_std']:.4f}",
            "Cov@95 (%)": f"{row['cov95_mean']:.1f} +/- {row['cov95_std']:.1f}",
            "Cov@99 (%)": f"{row['cov99_mean']:.1f} +/- {row['cov99_std']:.1f}",
        })
    
    display_df = pd.DataFrame(display_rows)
    print(display_df.to_string(index=False))
    
    print("\nRaw values:")
    for _, row in df.iterrows():
        print(f"  {row['method']:12s} | AURCC: {row['aurcc_mean']:.4f} | "
              f"Cov@95: {row['cov95_mean']:.1f}% | Cov@99: {row['cov99_mean']:.1f}%")

In [ ]:
# Cell 20
# Risk-Coverage Curve Plots 

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, backbone_name in zip(axes, BACKBONES):
    # SR: plot mean curve across seeds
    for method_name, results_dict, color, ls in [
        ("SR", sr_results[backbone_name], "gray", "--"),
        ("MCD", mcd_results[backbone_name], "blue", "--"),
        ("AquaSelect", aquaselect_results[backbone_name], "red", "-"),
    ]:
        # Average curves across seeds
        all_risks = []
        for seed in CFG["seeds"]:
            r = results_dict[seed]
            all_risks.append(r["risks"])
        mean_risks = np.mean(all_risks, axis=0)
        std_risks = np.std(all_risks, axis=0)
        coverages = results_dict[CFG["seeds"][0]]["coverages"]
        
        ax.plot(coverages, mean_risks, color=color, linestyle=ls, label=method_name, linewidth=2)
        ax.fill_between(coverages, mean_risks - std_risks, mean_risks + std_risks,
                        color=color, alpha=0.1)
    
    # DE: single curve
    de_r = de_results[backbone_name]
    ax.plot(de_r["coverages"], de_r["risks"], color="green", linestyle="--",
            label="DE", linewidth=2)
    
    ax.set_xlabel("Coverage", fontsize=12)
    ax.set_ylabel("Risk (Error Rate)", fontsize=12)
    ax.set_title(backbone_name.replace("_", " ").title(), fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, None])
    ax.grid(True, alpha=0.3)

plt.suptitle("Risk-Coverage Curves", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "risk_coverage_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: risk_coverage_curves.png")

In [ ]:
# Cell 21
# Per-seed detailed results

for backbone_name in BACKBONES:
    print(f"\n{'='*70}")
    print(f"  Per-Seed Results -- {backbone_name}")
    print(f"{'='*70}")
    
    rows = []
    for method_name, results_dict in [
        ("SR", sr_results[backbone_name]),
        ("MCD", mcd_results[backbone_name]),
        ("AquaSelect", aquaselect_results[backbone_name]),
    ]:
        for seed in CFG["seeds"]:
            r = results_dict[seed]
            rows.append({
                "Method": method_name,
                "Seed": seed,
                "Full Acc (%)": f"{r['full_acc']:.2f}",
                "AURCC": f"{r['aurcc']:.4f}",
                "Cov@95 (%)": f"{r['cov95']:.1f}",
                "Cov@99 (%)": f"{r['cov99']:.1f}",
            })
    
    # Add DE (single result)
    de_r = de_results[backbone_name]
    rows.append({
        "Method": "DE",
        "Seed": "all",
        "Full Acc (%)": f"{de_r['full_acc']:.2f}",
        "AURCC": f"{de_r['aurcc']:.4f}",
        "Cov@95 (%)": f"{de_r['cov95']:.1f}",
        "Cov@99 (%)": f"{de_r['cov99']:.1f}",
    })
    
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))

In [ ]:
# Cell 22
# Selective accuracy at different coverage levels 
# "If we only accept X% of predictions, what accuracy do we get?"

def selective_accuracy_at_coverage(predictions, targets, confidence, coverage_levels):
    """Compute selective accuracy at given coverage levels."""
    sorted_indices = np.argsort(-confidence)
    sorted_correct = (predictions == targets)[sorted_indices]
    n = len(sorted_correct)
    results = {}
    for cov in coverage_levels:
        k = max(1, int(cov * n))
        acc = sorted_correct[:k].mean() * 100
        results[cov] = acc
    return results


coverage_levels = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]

for backbone_name in BACKBONES:
    print(f"\n{'='*70}")
    print(f"  Selective Accuracy at Coverage -- {backbone_name} (seed 42)")
    print(f"{'='*70}")
    
    data = saved_logits[backbone_name]
    test_labels = data["test_labels"].numpy()
    seed = 42
    test_preds = data["test_logits"][seed].argmax(dim=1).numpy()
    
    # SR confidence
    sr_conf = F.softmax(data["test_logits"][seed], dim=1).max(dim=1).values.numpy()
    
    # MCD confidence
    mcd_r = mcd_results[backbone_name][seed]
    
    # DE confidence
    avg_probs = torch.zeros_like(F.softmax(data["test_logits"][CFG["seeds"][0]], dim=1))
    for s in CFG["seeds"]:
        avg_probs += F.softmax(data["test_logits"][s], dim=1)
    avg_probs /= len(CFG["seeds"])
    de_preds = avg_probs.argmax(dim=1).numpy()
    de_entropy = -(avg_probs * torch.log(avg_probs + 1e-8)).sum(dim=1)
    de_conf = -de_entropy.numpy()
    
    # AquaSelect confidence (stored in results)
    aq_conf = aquaselect_results[backbone_name][seed]["test_confidence"]
    
    # Print table
    header = f"{'Method':>12s}"
    for cov in coverage_levels:
        header += f" | {cov*100:5.0f}%"
    print(header)
    print("-" * len(header))
    
    for method_name, preds, conf in [
        ("SR", test_preds, sr_conf),
        ("DE", de_preds, de_conf),
        ("AquaSelect", test_preds, aq_conf),
    ]:
        accs = selective_accuracy_at_coverage(preds, test_labels, conf, coverage_levels)
        row = f"{method_name:>12s}"
        for cov in coverage_levels:
            row += f" | {accs[cov]:5.1f}%"
        print(row)
    
    print(f"\n  At 100% coverage = standard accuracy (no abstention).")
    print(f"  At lower coverage, AquaSelect should show highest accuracy (best selection).")

In [ ]:
# Cell 23
# Save all results (ablation + explainability) 

output_dir = CFG["output_dir"]

# Prepare saveable results (remove non-serializable objects)
save_data = {
    "sr_results": {},
    "mcd_results": {},
    "de_results": {},
    "aquaselect_results": {},
    "udas_features": {
        "val_visibility": val_visibility,
        "val_color_cast_norm": val_color_cast_norm,
        "test_visibility": test_visibility,
        "test_color_cast_norm": test_color_cast_norm,
        "cc_99": cc_99,
    },
    "config": CFG,
    "label_names": label_names,
}

# SR results
for backbone_name in BACKBONES:
    save_data["sr_results"][backbone_name] = {}
    for seed in CFG["seeds"]:
        r = sr_results[backbone_name][seed]
        save_data["sr_results"][backbone_name][seed] = {
            "aurcc": r["aurcc"], "cov95": r["cov95"], "cov99": r["cov99"],
            "full_acc": r["full_acc"],
            "coverages": r["coverages"], "risks": r["risks"],
        }

# MCD results
for backbone_name in BACKBONES:
    save_data["mcd_results"][backbone_name] = {}
    for seed in CFG["seeds"]:
        r = mcd_results[backbone_name][seed]
        save_data["mcd_results"][backbone_name][seed] = {
            "aurcc": r["aurcc"], "cov95": r["cov95"], "cov99": r["cov99"],
            "full_acc": r["full_acc"],
            "coverages": r["coverages"], "risks": r["risks"],
        }

# DE results
for backbone_name in BACKBONES:
    r = de_results[backbone_name]
    save_data["de_results"][backbone_name] = {
        "aurcc": r["aurcc"], "cov95": r["cov95"], "cov99": r["cov99"],
        "full_acc": r["full_acc"],
        "coverages": r["coverages"], "risks": r["risks"],
    }

# AquaSelect results
for backbone_name in BACKBONES:
    save_data["aquaselect_results"][backbone_name] = {}
    for seed in CFG["seeds"]:
        r = aquaselect_results[backbone_name][seed]
        save_data["aquaselect_results"][backbone_name][seed] = {
            "aurcc": r["aurcc"], "cov95": r["cov95"], "cov99": r["cov99"],
            "full_acc": r["full_acc"],
            "coverages": r["coverages"], "risks": r["risks"],
            "test_confidence": r["test_confidence"],
            "test_sel_scores": r["test_sel_scores"],
            "val_sel_scores": r["val_sel_scores"],
            "temperature": r["temperature"],
        }

results_path = os.path.join(output_dir, "notebook2_all_results.pth")
torch.save(save_data, results_path)
print(f"Saved all results: {results_path} ({os.path.getsize(results_path) / 1e6:.1f} MB)")

# save the summary tables as CSV
for backbone_name in BACKBONES:
    rows = []
    for method_name, rd in [("SR", sr_results[backbone_name]),
                             ("MCD", mcd_results[backbone_name]),
                             ("AquaSelect", aquaselect_results[backbone_name])]:
        for seed in CFG["seeds"]:
            r = rd[seed]
            rows.append({"method": method_name, "seed": seed,
                         "aurcc": r["aurcc"], "cov95": r["cov95"], "cov99": r["cov99"]})
    r = de_results[backbone_name]
    rows.append({"method": "DE", "seed": "all",
                 "aurcc": r["aurcc"], "cov95": r["cov95"], "cov99": r["cov99"]})
    
    csv_path = os.path.join(output_dir, f"selective_results_{backbone_name}.csv")
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")

# List outputs
print(f"\nAll outputs in {output_dir}:")
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        print(f"  {f} ({os.path.getsize(fpath) / 1e6:.1f} MB)")

In [ ]:
# Cell 24
# Done 
print("Notebook 2 complete.")